# 1.2 · El chat que se olvida de ti

**Objetivo:** vivir en primera persona que la API es *stateless*: no recuerda nada entre llamadas.

**Al terminar tendrás:** una función `chat()` con historial y streaming, y la lista `messages` inspeccionada: esa lista **es** la memoria.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import anthropic
client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

In [ ]:
SYSTEM = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, en tono cercano y como máximo en dos frases."
)
    

## Versión ingenua (a propósito)

La forma más intuitiva de hacer un chat: una función que envía la pregunta y devuelve el texto. **Va a fallar.**

In [ ]:
# ❌ FALLO PROVOCADO
def chat_ingenuo(pregunta):
    response = client.messages.create(
        model=MODEL, max_tokens=1024, system=SYSTEM,
        messages=[{"role": "user", "content": pregunta}],
    )
    return next(b.text for b in response.content if b.type == "text")

In [ ]:
chat_ingenuo("Hola, me llamo Ana y tengo un problema con la factura 1234")

In [ ]:
chat_ingenuo("¿Cómo me llamo y qué factura era?")

## Por qué falla

Las dos celdas están una encima de la otra, pero para la API eso no significa nada: **solo cuenta lo que se envía en cada petición**.

```
 llamada 1:  [user: "Hola, me llamo Ana… factura 1234"]          → respuesta
 llamada 2:  [user: "¿Cómo me llamo y qué factura era?"]          → "No lo sé"
              ↑ el servidor no guarda sesión: esta petición llega sola
```

- Cada `create` es una petición HTTP aislada.
- La "memoria" de un chat es **el cliente reenviando toda la conversación** en cada turno:

```
 turno 1:  [user₁]                               → assistant₁
 turno 2:  [user₁, assistant₁, user₂]            → assistant₂
 turno 3:  [user₁, assistant₁, user₂, assistant₂, user₃] → …
```

- Corolario: **en cada turno pagas de nuevo todo lo anterior**.

## El arreglo: el historial

Una lista `messages` que vive fuera de la función. En cada turno:
1. añadimos el mensaje del usuario,
2. enviamos **la lista entera**,
3. añadimos la respuesta del asistente.

Fíjate en que se guarda `response.content` **completo**, no solo el texto. Es como hay que hacerlo, en el bloque 2 será obligatoria.

In [ ]:
messages = []

def chat(pregunta):
    messages.append({"role": "user", "content": pregunta})
    response = client.messages.create(
        model=MODEL, max_tokens=1024, system=SYSTEM, messages=messages,
    )
    messages.append({"role": "assistant", "content": response.content})
    print(f"[input_tokens={response.usage.input_tokens}]")
    return next(b.text for b in response.content if b.type == "text")

Una celda por turno:

In [ ]:
chat("Hola, me llamo Ana y tengo un problema con la factura 1234")

In [ ]:
chat("¿Cómo me llamo y qué factura era?")

### Esta es la memoria

Lo que se envía en cada llamada, tal cual. Fíjate en los mensajes `assistant`: guardan todos los bloques (también `thinking`, si lo hay).

> Abre también el panel **Variables** de la barra del notebook: `messages` aparece con su longitud, que sube en cada turno.

In [ ]:
messages

Unos cuantos turnos más. Mira cómo crece `input_tokens` en cada celda:

In [ ]:
chat("El servidor está bloqueado, no puedo acceder a él")

In [ ]:
chat("Lo he probado en Chrome y en Firefox, pasa en los dos")

In [ ]:
chat("¿Me puedes resumir el caso en una línea para pasarlo a nivel 2?")

### Olvidar a propósito

Si vaciamos la lista, el modelo "olvida". No ha cambiado nada en el servidor: hemos borrado **nuestro** historial.

In [ ]:
messages = []

In [ ]:
chat("¿Cómo me llamo?")

## Streaming

Problema: con respuestas largas, la celda se queda en `[*]` sin mostrar nada hasta el final.

Con `client.messages.stream` el texto llega a trozos y lo imprimimos según llega. Al terminar, `get_final_message()` da el mensaje completo, **que hay que guardar en el historial igual que antes**.

El streaming cambia la **experiencia**, no el modelo mental.

In [ ]:
def chat(pregunta):
    messages.append({"role": "user", "content": pregunta})
    with client.messages.stream(
        model=MODEL, max_tokens=4096, system=SYSTEM, messages=messages,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final = stream.get_final_message()
    messages.append({"role": "assistant", "content": final.content})
    print(f"\n[input_tokens={final.usage.input_tokens}]")

In [ ]:
chat("Explícame paso a paso cómo exportar un informe a PDF")

## Resumen

- La API no recuerda nada: cada llamada es independiente.
- El historial es tuyo: es la lista `messages`. Si la pierdes, el modelo lo pierde.
- Cada turno reenvía y paga todo lo anterior: si el historial crece, tu factura crece.
- Se guarda el `content` completo de la respuesta, no solo el texto.

**Para el siguiente vídeo:** ¿qué hacemos cuando el historial crece demasiado?

## 🔸 Reto

Escribe una función `resumen_historial()` que imprima cuántos mensajes hay y de qué rol es cada uno.